In [1]:
from pathlib import Path
import os
import json
import platform
import sys

PROJECT_ROOT = Path("/home/ronie/Programs/SWARM_DRONES")
DATASETS_DIR = PROJECT_ROOT / "datasets"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"
TRAINING_DIR = OUTPUTS_DIR / "training"
INFERENCE_DIR = OUTPUTS_DIR / "inference"

for directory in [DATASETS_DIR, OUTPUTS_DIR, TRAINING_DIR, INFERENCE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SURVEILLANCE_TAXONOMY = [
    "person",
    "pedestrian",
    "crowd",
    "car",
    "van",
    "truck",
    "bus",
    "vehicle",
    "smoke",
    "fire",
    "damaged_structure",
    "obstacle",
    "restricted_zone_object",
    "unknown_heavy_object",
    "infrastructure",
    "building",
    "container",
    "restricted_equipment_like_object",
    "hazardous_object_for_review",
    "unidentified_vehicle",
]

SAFETY_BOUNDARY = {
    "project_type": "UAV surveillance and human-review event reporting",
    "autonomous_targeting": False,
    "weapon_control": False,
    "engagement_decision": False,
    "human_review_required": True,
}

print("Project root:", PROJECT_ROOT)
print("Datasets directory:", DATASETS_DIR)
print("Training output directory:", TRAINING_DIR)
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Platform:", platform.platform())

print("\nSurveillance taxonomy:")
for index, class_name in enumerate(SURVEILLANCE_TAXONOMY):
    print(f"{index:02d}: {class_name}")

print("\nSafety boundary:")
print(json.dumps(SAFETY_BOUNDARY, indent=2))

Project root: /home/ronie/Programs/SWARM_DRONES
Datasets directory: /home/ronie/Programs/SWARM_DRONES/datasets
Training output directory: /home/ronie/Programs/SWARM_DRONES/outputs/training
Python executable: /home/ronie/Programs/SWARM_DRONES/.venv/bin/python
Python version: 3.10.12 (main, Mar  3 2026, 11:56:32) [GCC 11.4.0]
Platform: Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.35

Surveillance taxonomy:
00: person
01: pedestrian
02: crowd
03: car
04: van
05: truck
06: bus
07: vehicle
08: smoke
09: fire
10: damaged_structure
11: obstacle
12: restricted_zone_object
13: unknown_heavy_object
14: infrastructure
15: building
16: container
17: restricted_equipment_like_object
18: hazardous_object_for_review
19: unidentified_vehicle

Safety boundary:
{
  "project_type": "UAV surveillance and human-review event reporting",
  "autonomous_targeting": false,
  "weapon_control": false,
  "engagement_decision": false,
  "human_review_required": true
}


In [2]:
import sys
import torch
import ultralytics
from ultralytics import YOLO

print("Python executable:")
print(sys.executable)

print("\nPyTorch version:")
print(torch.__version__)

print("\nUltralytics version:")
print(ultralytics.__version__)

print("\nCUDA available:")
print(torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = 0
    print("\nGPU name:")
    print(torch.cuda.get_device_name(0))

    print("\nCUDA version used by PyTorch:")
    print(torch.version.cuda)

    total_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print("\nTotal GPU memory:")
    print(round(total_memory_gb, 2), "GB")
else:
    DEVICE = "cpu"
    print("\nGPU not available. Training will run on CPU, which will be slow.")

print("\nSelected training device:")
print(DEVICE)

# Load YOLO pretrained model only to verify that Ultralytics is working.
model = YOLO("yolov8n.pt")
print("\nYOLOv8n loaded successfully.")

Python executable:
/home/ronie/Programs/SWARM_DRONES/.venv/bin/python

PyTorch version:
2.12.1+cu130

Ultralytics version:
8.4.80

CUDA available:
True

GPU name:
NVIDIA GeForce RTX 3070 Ti Laptop GPU

CUDA version used by PyTorch:
13.0

Total GPU memory:
8.0 GB

Selected training device:
0

YOLOv8n loaded successfully.


In [3]:
from pathlib import Path
import yaml
import ultralytics

ULTRALYTICS_ROOT = Path(ultralytics.__file__).parent
VISDRONE_YAML_PATH = ULTRALYTICS_ROOT / "cfg" / "datasets" / "VisDrone.yaml"

print("Ultralytics root:")
print(ULTRALYTICS_ROOT)

print("\nVisDrone YAML path:")
print(VISDRONE_YAML_PATH)

if not VISDRONE_YAML_PATH.exists():
    raise FileNotFoundError(f"VisDrone.yaml not found at: {VISDRONE_YAML_PATH}")

with VISDRONE_YAML_PATH.open("r", encoding="utf-8") as file:
    visdrone_yaml = yaml.safe_load(file)

print("\nVisDrone YAML keys:")
print(list(visdrone_yaml.keys()))

print("\nOriginal VisDrone class names:")
visdrone_names = visdrone_yaml["names"]

for class_id, class_name in visdrone_names.items():
    print(f"{class_id}: {class_name}")

# Mapping VisDrone classes into our safe surveillance taxonomy.
VISDRONE_TO_SURVEILLANCE_MAP = {
    "pedestrian": "pedestrian",
    "people": "person",
    "bicycle": "vehicle",
    "car": "car",
    "van": "van",
    "truck": "truck",
    "tricycle": "vehicle",
    "awning-tricycle": "vehicle",
    "bus": "bus",
    "motor": "vehicle",
}

print("\nVisDrone → Surveillance taxonomy mapping:")
for source_class, target_class in VISDRONE_TO_SURVEILLANCE_MAP.items():
    print(f"{source_class:18s} → {target_class}")

STAGE_1_SUPPORTED_CLASSES = sorted(set(VISDRONE_TO_SURVEILLANCE_MAP.values()))

print("\nStage 1 supported surveillance classes:")
for class_name in STAGE_1_SUPPORTED_CLASSES:
    print("-", class_name)

TRAINING_CONFIG = {
    "stage": "stage_1_visdrone_person_vehicle",
    "base_model": "yolov8n.pt",
    "data_yaml": "VisDrone.yaml",
    "epochs_first_test": 3,
    "epochs_full_run": 20,
    "image_size": 640,
    "batch_size": 8,
    "device": DEVICE,
    "workers": 2,
    "project": str(TRAINING_DIR),
    "name_first_test": "yolov8n_visdrone_smoke_test",
    "name_full_run": "yolov8n_visdrone_full_run",
}

print("\nTraining configuration:")
for key, value in TRAINING_CONFIG.items():
    print(f"{key}: {value}")

Ultralytics root:
/home/ronie/Programs/SWARM_DRONES/.venv/lib/python3.10/site-packages/ultralytics

VisDrone YAML path:
/home/ronie/Programs/SWARM_DRONES/.venv/lib/python3.10/site-packages/ultralytics/cfg/datasets/VisDrone.yaml

VisDrone YAML keys:
['path', 'train', 'val', 'test', 'names', 'download']

Original VisDrone class names:
0: pedestrian
1: people
2: bicycle
3: car
4: van
5: truck
6: tricycle
7: awning-tricycle
8: bus
9: motor

VisDrone → Surveillance taxonomy mapping:
pedestrian         → pedestrian
people             → person
bicycle            → vehicle
car                → car
van                → van
truck              → truck
tricycle           → vehicle
awning-tricycle    → vehicle
bus                → bus
motor              → vehicle

Stage 1 supported surveillance classes:
- bus
- car
- pedestrian
- person
- truck
- van
- vehicle

Training configuration:
stage: stage_1_visdrone_person_vehicle
base_model: yolov8n.pt
data_yaml: VisDrone.yaml
epochs_first_test: 3
epochs_

In [4]:
from ultralytics import YOLO, settings
from pathlib import Path
import time
import torch

# Keep Ultralytics datasets and runs inside our project folder.
settings.update({
    "datasets_dir": str(DATASETS_DIR),
    "runs_dir": str(OUTPUTS_DIR / "ultralytics_runs"),
})

print("Ultralytics datasets directory:")
print(settings["datasets_dir"])

print("\nUltralytics runs directory:")
print(settings["runs_dir"])

print("\nCUDA available before training:")
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

SMOKE_TEST_NAME = "yolov8n_visdrone_smoke_test"

model = YOLO("yolov8n.pt")

start_time = time.time()

results = model.train(
    data="VisDrone.yaml",
    epochs=1,
    imgsz=640,
    batch=8,
    device=DEVICE,
    workers=2,
    project=str(TRAINING_DIR),
    name=SMOKE_TEST_NAME,
    exist_ok=True,
    patience=1,
    plots=True,
    verbose=True,
)

end_time = time.time()

smoke_test_dir = TRAINING_DIR / SMOKE_TEST_NAME
best_model_path = smoke_test_dir / "weights" / "best.pt"
last_model_path = smoke_test_dir / "weights" / "last.pt"

print("\nSmoke test training finished.")
print("Training time minutes:", round((end_time - start_time) / 60, 2))
print("Smoke test output directory:", smoke_test_dir)
print("best.pt exists:", best_model_path.exists())
print("last.pt exists:", last_model_path.exists())

if torch.cuda.is_available():
    print("\nGPU memory allocated after training:")
    print(round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")
    print("GPU memory reserved after training:")
    print(round(torch.cuda.memory_reserved(0) / 1024**3, 2), "GB")

Ultralytics datasets directory:
/home/ronie/Programs/SWARM_DRONES/datasets

Ultralytics runs directory:
/home/ronie/Programs/SWARM_DRONES/outputs/ultralytics_runs

CUDA available before training:
True
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU
Ultralytics 8.4.80 🚀 Python-3.10.12 torch-2.12.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, li

In [5]:
from pathlib import Path
import pandas as pd

SMOKE_TEST_NAME = "yolov8n_visdrone_smoke_test"
smoke_test_dir = TRAINING_DIR / SMOKE_TEST_NAME

print("Smoke test directory:")
print(smoke_test_dir)

print("\nDirectory exists:")
print(smoke_test_dir.exists())

print("\nMain files:")
for path in sorted(smoke_test_dir.iterdir()):
    print("-", path.name)

weights_dir = smoke_test_dir / "weights"
print("\nWeights directory:")
print(weights_dir)

print("\nWeights files:")
for path in sorted(weights_dir.iterdir()):
    size_mb = path.stat().st_size / 1024**2
    print(f"- {path.name}: {size_mb:.2f} MB")

results_csv = smoke_test_dir / "results.csv"

print("\nresults.csv exists:")
print(results_csv.exists())

if results_csv.exists():
    df = pd.read_csv(results_csv)
    display(df.tail())

    print("\nAvailable metric columns:")
    for column in df.columns:
        print("-", column)
else:
    print("results.csv not found.")

Smoke test directory:
/home/ronie/Programs/SWARM_DRONES/outputs/training/yolov8n_visdrone_smoke_test

Directory exists:
True

Main files:
- BoxF1_curve.png
- BoxPR_curve.png
- BoxP_curve.png
- BoxR_curve.png
- args.yaml
- confusion_matrix.png
- confusion_matrix_normalized.png
- labels.jpg
- results.csv
- results.png
- train_batch0.jpg
- train_batch1.jpg
- train_batch2.jpg
- val_batch0_labels.jpg
- val_batch0_pred.jpg
- val_batch1_labels.jpg
- val_batch1_pred.jpg
- val_batch2_labels.jpg
- val_batch2_pred.jpg
- weights

Weights directory:
/home/ronie/Programs/SWARM_DRONES/outputs/training/yolov8n_visdrone_smoke_test/weights

Weights files:
- best.pt: 5.93 MB
- last.pt: 5.93 MB

results.csv exists:
True


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
0,1,78.5336,1.97392,2.33058,1.0289,0.28911,0.15756,0.1055,0.05447,1.7534,1.58984,0.98587,0.000238,0.000238,0.000238



Available metric columns:
- epoch
- time
- train/box_loss
- train/cls_loss
- train/dfl_loss
- metrics/precision(B)
- metrics/recall(B)
- metrics/mAP50(B)
- metrics/mAP50-95(B)
- val/box_loss
- val/cls_loss
- val/dfl_loss
- lr/pg0
- lr/pg1
- lr/pg2


In [6]:
from ultralytics import YOLO
from pathlib import Path
from collections import Counter

SMOKE_TEST_NAME = "yolov8n_visdrone_smoke_test"

smoke_test_dir = TRAINING_DIR / SMOKE_TEST_NAME
best_model_path = smoke_test_dir / "weights" / "best.pt"

camera_frames_dir = PROJECT_ROOT / "outputs" / "camera_frames" / "raw"

inference_output_dir = INFERENCE_DIR
inference_run_name = "visdrone_smoke_test_on_gazebo_frames"

print("Best model path:")
print(best_model_path)

print("\nBest model exists:")
print(best_model_path.exists())

print("\nCamera frames directory:")
print(camera_frames_dir)

print("\nCamera frames directory exists:")
print(camera_frames_dir.exists())

if not best_model_path.exists():
    raise FileNotFoundError(f"Best model not found: {best_model_path}")

if not camera_frames_dir.exists():
    raise FileNotFoundError(
        f"Camera frames not found: {camera_frames_dir}. "
        "Capture Gazebo camera frames first before running inference."
    )

image_files = sorted(
    list(camera_frames_dir.glob("*.png")) +
    list(camera_frames_dir.glob("*.jpg")) +
    list(camera_frames_dir.glob("*.jpeg"))
)

print("\nNumber of camera frames found:")
print(len(image_files))

if len(image_files) == 0:
    raise FileNotFoundError(f"No image files found inside: {camera_frames_dir}")

trained_model = YOLO(str(best_model_path))

results = trained_model.predict(
    source=str(camera_frames_dir),
    imgsz=640,
    conf=0.25,
    device=DEVICE,
    save=True,
    save_txt=True,
    save_conf=True,
    project=str(inference_output_dir),
    name=inference_run_name,
    exist_ok=True,
    verbose=True,
)

class_counter = Counter()

for result in results:
    for box in result.boxes:
        class_id = int(box.cls[0].item())
        class_name = result.names[class_id]
        class_counter[class_name] += 1

saved_result_dir = inference_output_dir / inference_run_name

print("\nInference complete.")
print("Saved result directory:")
print(saved_result_dir)

print("\nDetected class counts:")
if class_counter:
    for class_name, count in class_counter.items():
        print(f"- {class_name}: {count}")
else:
    print("No detections at confidence threshold 0.25")

print("\nOpen this folder to view annotated images:")
print(saved_result_dir)

Best model path:
/home/ronie/Programs/SWARM_DRONES/outputs/training/yolov8n_visdrone_smoke_test/weights/best.pt

Best model exists:
True

Camera frames directory:
/home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw

Camera frames directory exists:
True

Number of camera frames found:
10

image 1/10 /home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw/camera_frame_0000.png: 480x640 (no detections), 51.5ms
image 2/10 /home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw/camera_frame_0001.png: 480x640 (no detections), 16.8ms
image 3/10 /home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw/camera_frame_0002.png: 480x640 (no detections), 16.3ms
image 4/10 /home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw/camera_frame_0003.png: 480x640 (no detections), 16.6ms
image 5/10 /home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw/camera_frame_0004.png: 480x640 (no detections), 16.5ms
image 6/10 /home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw/ca

In [7]:
from ultralytics import YOLO
from pathlib import Path
from collections import Counter

camera_frames_dir = PROJECT_ROOT / "outputs" / "camera_frames" / "raw"

models_to_test = {
    "coco_yolov8n_baseline": "yolov8n.pt",
    "visdrone_1epoch_smoke_test": str(TRAINING_DIR / "yolov8n_visdrone_smoke_test" / "weights" / "best.pt"),
}

for run_name, model_path in models_to_test.items():
    print("\n" + "=" * 80)
    print("Testing model:", run_name)
    print("Model path:", model_path)

    model = YOLO(model_path)

    results = model.predict(
        source=str(camera_frames_dir),
        imgsz=640,
        conf=0.15,
        device=DEVICE,
        save=True,
        save_txt=True,
        save_conf=True,
        project=str(INFERENCE_DIR),
        name=run_name,
        exist_ok=True,
        verbose=False,
    )

    class_counter = Counter()

    for result in results:
        for box in result.boxes:
            class_id = int(box.cls[0].item())
            class_name = result.names[class_id]
            class_counter[class_name] += 1

    print("Detected class counts at conf=0.15:")
    if class_counter:
        for class_name, count in class_counter.items():
            print(f"- {class_name}: {count}")
    else:
        print("No detections")

    print("Saved to:", INFERENCE_DIR / run_name)


Testing model: coco_yolov8n_baseline
Model path: yolov8n.pt
Results saved to /home/ronie/Programs/SWARM_DRONES/outputs/inference/coco_yolov8n_baseline
10 labels saved to /home/ronie/Programs/SWARM_DRONES/outputs/inference/coco_yolov8n_baseline/labels
Detected class counts at conf=0.15:
- person: 20
- truck: 10
- bus: 10
- car: 40
Saved to: /home/ronie/Programs/SWARM_DRONES/outputs/inference/coco_yolov8n_baseline

Testing model: visdrone_1epoch_smoke_test
Model path: /home/ronie/Programs/SWARM_DRONES/outputs/training/yolov8n_visdrone_smoke_test/weights/best.pt
Results saved to /home/ronie/Programs/SWARM_DRONES/outputs/inference/visdrone_1epoch_smoke_test
10 labels saved to /home/ronie/Programs/SWARM_DRONES/outputs/inference/visdrone_1epoch_smoke_test/labels
Detected class counts at conf=0.15:
- car: 40
Saved to: /home/ronie/Programs/SWARM_DRONES/outputs/inference/visdrone_1epoch_smoke_test


In [8]:
from ultralytics import YOLO, settings
from pathlib import Path
import time
import torch

settings.update({
    "datasets_dir": str(DATASETS_DIR),
    "runs_dir": str(OUTPUTS_DIR / "ultralytics_runs"),
})

FULL_RUN_NAME = "yolov8n_visdrone_full_run"
full_run_dir = TRAINING_DIR / FULL_RUN_NAME

print("Full training output directory:")
print(full_run_dir)

print("\nCUDA available:")
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Initial GPU memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")
    print("Initial GPU memory reserved:", round(torch.cuda.memory_reserved(0) / 1024**3, 2), "GB")

print("\nTraining configuration:")
print("Model: yolov8n.pt")
print("Dataset: VisDrone.yaml")
print("Epochs: 50")
print("Image size: 640")
print("Batch size: 8")
print("Device:", DEVICE)

model = YOLO("yolov8n.pt")

start_time = time.time()

results = model.train(
    data="VisDrone.yaml",
    epochs=50,
    imgsz=640,
    batch=8,
    device=DEVICE,
    workers=2,
    project=str(TRAINING_DIR),
    name=FULL_RUN_NAME,
    exist_ok=True,
    patience=7,
    plots=True,
    save=True,
    save_period=5,
    verbose=True,
)

end_time = time.time()

best_model_path = full_run_dir / "weights" / "best.pt"
last_model_path = full_run_dir / "weights" / "last.pt"
results_csv_path = full_run_dir / "results.csv"

print("\nFull training finished.")
print("Training time minutes:", round((end_time - start_time) / 60, 2))
print("Full run output directory:", full_run_dir)
print("best.pt exists:", best_model_path.exists())
print("last.pt exists:", last_model_path.exists())
print("results.csv exists:", results_csv_path.exists())

if torch.cuda.is_available():
    print("\nGPU memory allocated after training:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")
    print("GPU memory reserved after training:", round(torch.cuda.memory_reserved(0) / 1024**3, 2), "GB")

Full training output directory:
/home/ronie/Programs/SWARM_DRONES/outputs/training/yolov8n_visdrone_full_run

CUDA available:
True
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU
Initial GPU memory allocated: 0.12 GB
Initial GPU memory reserved: 0.21 GB

Training configuration:
Model: yolov8n.pt
Dataset: VisDrone.yaml
Epochs: 50
Image size: 640
Batch size: 8
Device: 0
Ultralytics 8.4.80 🚀 Python-3.10.12 torch-2.12.1+cu130 CUDA:0 (NVIDIA GeForce RTX 3070 Ti Laptop GPU, 8192MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=VisDrone.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, form

In [9]:
from pathlib import Path
import shutil
import json
from datetime import datetime

FULL_RUN_NAME = "yolov8n_visdrone_full_run"
full_run_dir = TRAINING_DIR / FULL_RUN_NAME

best_model_path = full_run_dir / "weights" / "best.pt"
last_model_path = full_run_dir / "weights" / "last.pt"
results_csv_path = full_run_dir / "results.csv"
args_yaml_path = full_run_dir / "args.yaml"

MODEL_REGISTRY_DIR = PROJECT_ROOT / "models" / "visdrone"
MODEL_REGISTRY_DIR.mkdir(parents=True, exist_ok=True)

registered_best_model = MODEL_REGISTRY_DIR / "yolov8n_visdrone_full_run_best.pt"
registered_last_model = MODEL_REGISTRY_DIR / "yolov8n_visdrone_full_run_last.pt"
registered_results_csv = MODEL_REGISTRY_DIR / "yolov8n_visdrone_full_run_results.csv"
registered_args_yaml = MODEL_REGISTRY_DIR / "yolov8n_visdrone_full_run_args.yaml"
registered_metadata_json = MODEL_REGISTRY_DIR / "yolov8n_visdrone_full_run_metadata.json"

if not best_model_path.exists():
    raise FileNotFoundError(f"best.pt not found: {best_model_path}")

if not last_model_path.exists():
    raise FileNotFoundError(f"last.pt not found: {last_model_path}")

shutil.copy2(best_model_path, registered_best_model)
shutil.copy2(last_model_path, registered_last_model)

if results_csv_path.exists():
    shutil.copy2(results_csv_path, registered_results_csv)

if args_yaml_path.exists():
    shutil.copy2(args_yaml_path, registered_args_yaml)

metadata = {
    "model_name": "yolov8n_visdrone_full_run",
    "base_model": "yolov8n.pt",
    "dataset": "VisDrone",
    "purpose": "UAV surveillance person and vehicle detection for human-review reporting",
    "training_output_dir": str(full_run_dir),
    "registered_best_model": str(registered_best_model),
    "registered_last_model": str(registered_last_model),
    "registered_at": datetime.now().isoformat(),
    "safe_usage": {
        "autonomous_targeting": False,
        "weapon_control": False,
        "engagement_decision": False,
        "human_review_required": True,
    },
    "classes": [
        "pedestrian",
        "people",
        "bicycle",
        "car",
        "van",
        "truck",
        "tricycle",
        "awning-tricycle",
        "bus",
        "motor",
    ],
}

registered_metadata_json.write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Model weights saved permanently.")

print("\nBest model:")
print(registered_best_model)

print("\nLast model:")
print(registered_last_model)

print("\nMetadata:")
print(registered_metadata_json)

print("\nFiles in model registry:")
for path in sorted(MODEL_REGISTRY_DIR.iterdir()):
    size_mb = path.stat().st_size / 1024**2
    print(f"- {path.name}: {size_mb:.2f} MB")
    

Model weights saved permanently.

Best model:
/home/ronie/Programs/SWARM_DRONES/models/visdrone/yolov8n_visdrone_full_run_best.pt

Last model:
/home/ronie/Programs/SWARM_DRONES/models/visdrone/yolov8n_visdrone_full_run_last.pt

Metadata:
/home/ronie/Programs/SWARM_DRONES/models/visdrone/yolov8n_visdrone_full_run_metadata.json

Files in model registry:
- yolov8n_visdrone_full_run_args.yaml: 0.00 MB
- yolov8n_visdrone_full_run_best.pt: 5.94 MB
- yolov8n_visdrone_full_run_last.pt: 5.94 MB
- yolov8n_visdrone_full_run_metadata.json: 0.00 MB
- yolov8n_visdrone_full_run_results.csv: 0.01 MB


In [1]:
from pathlib import Path
from ultralytics import YOLO
import torch

PROJECT_ROOT = Path("/home/ronie/Programs/SWARM_DRONES")

REGISTERED_VISDRONE_MODEL = (
    PROJECT_ROOT
    / "models"
    / "visdrone"
    / "yolov8n_visdrone_full_run_best.pt"
)

print("Registered VisDrone model:")
print(REGISTERED_VISDRONE_MODEL)

print("\nModel exists:")
print(REGISTERED_VISDRONE_MODEL.exists())

if not REGISTERED_VISDRONE_MODEL.exists():
    raise FileNotFoundError(f"Model not found: {REGISTERED_VISDRONE_MODEL}")

device = 0 if torch.cuda.is_available() else "cpu"

print("\nSelected device:")
print(device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model = YOLO(str(REGISTERED_VISDRONE_MODEL))

print("\nModel loaded successfully.")

print("\nModel class names:")
for class_id, class_name in model.names.items():
    print(f"{class_id}: {class_name}")

Registered VisDrone model:
/home/ronie/Programs/SWARM_DRONES/models/visdrone/yolov8n_visdrone_full_run_best.pt

Model exists:
True

Selected device:
0
GPU: NVIDIA GeForce RTX 3070 Ti Laptop GPU

Model loaded successfully.

Model class names:
0: pedestrian
1: people
2: bicycle
3: car
4: van
5: truck
6: tricycle
7: awning-tricycle
8: bus
9: motor


In [2]:
from pathlib import Path
from ultralytics import YOLO
from collections import Counter
import torch

PROJECT_ROOT = Path("/home/ronie/Programs/SWARM_DRONES")

REGISTERED_VISDRONE_MODEL = (
    PROJECT_ROOT
    / "models"
    / "visdrone"
    / "yolov8n_visdrone_full_run_best.pt"
)

CAMERA_FRAMES_DIR = PROJECT_ROOT / "outputs" / "camera_frames" / "raw"
INFERENCE_DIR = PROJECT_ROOT / "outputs" / "inference"
RUN_NAME = "registered_visdrone_on_gazebo_frames"

device = 0 if torch.cuda.is_available() else "cpu"

print("Model:")
print(REGISTERED_VISDRONE_MODEL)

print("\nCamera frames directory:")
print(CAMERA_FRAMES_DIR)

print("\nCamera frames exist:")
print(CAMERA_FRAMES_DIR.exists())

if not CAMERA_FRAMES_DIR.exists():
    raise FileNotFoundError(f"Camera frames directory not found: {CAMERA_FRAMES_DIR}")

image_files = sorted(
    list(CAMERA_FRAMES_DIR.glob("*.png")) +
    list(CAMERA_FRAMES_DIR.glob("*.jpg")) +
    list(CAMERA_FRAMES_DIR.glob("*.jpeg"))
)

print("\nNumber of frames found:")
print(len(image_files))

if len(image_files) == 0:
    raise FileNotFoundError("No camera frames found.")

model = YOLO(str(REGISTERED_VISDRONE_MODEL))

safe_label_map = {
    "pedestrian": "pedestrian",
    "people": "person",
    "bicycle": "vehicle",
    "car": "car",
    "van": "van",
    "truck": "truck",
    "tricycle": "vehicle",
    "awning-tricycle": "vehicle",
    "bus": "bus",
    "motor": "vehicle",
}

results = model.predict(
    source=str(CAMERA_FRAMES_DIR),
    imgsz=640,
    conf=0.15,
    device=device,
    save=True,
    save_txt=True,
    save_conf=True,
    project=str(INFERENCE_DIR),
    name=RUN_NAME,
    exist_ok=True,
    verbose=True,
)

original_counts = Counter()
safe_counts = Counter()

for result in results:
    for box in result.boxes:
        class_id = int(box.cls[0].item())
        original_name = result.names[class_id]
        safe_name = safe_label_map.get(original_name, "object_for_review")

        original_counts[original_name] += 1
        safe_counts[safe_name] += 1

saved_dir = INFERENCE_DIR / RUN_NAME

print("\nInference complete.")
print("Saved directory:")
print(saved_dir)

print("\nOriginal VisDrone class counts:")
if original_counts:
    for name, count in original_counts.items():
        print(f"- {name}: {count}")
else:
    print("No detections")

print("\nSafe surveillance label counts:")
if safe_counts:
    for name, count in safe_counts.items():
        print(f"- {name}: {count}")
else:
    print("No detections")

print("\nOpen result folder:")
print(saved_dir)

Model:
/home/ronie/Programs/SWARM_DRONES/models/visdrone/yolov8n_visdrone_full_run_best.pt

Camera frames directory:
/home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw

Camera frames exist:
True

Number of frames found:
10

image 1/10 /home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw/camera_frame_0000.png: 480x640 (no detections), 66.3ms
image 2/10 /home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw/camera_frame_0001.png: 480x640 (no detections), 6.8ms
image 3/10 /home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw/camera_frame_0002.png: 480x640 (no detections), 4.2ms
image 4/10 /home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw/camera_frame_0003.png: 480x640 (no detections), 4.6ms
image 5/10 /home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw/camera_frame_0004.png: 480x640 (no detections), 4.2ms
image 6/10 /home/ronie/Programs/SWARM_DRONES/outputs/camera_frames/raw/camera_frame_0005.png: 480x640 (no detections), 4.7ms
image 7/10 /home

In [3]:
from pathlib import Path
from ultralytics import YOLO
from collections import Counter
import torch

PROJECT_ROOT = Path("/home/ronie/Programs/SWARM_DRONES")

REGISTERED_VISDRONE_MODEL = (
    PROJECT_ROOT
    / "models"
    / "visdrone"
    / "yolov8n_visdrone_full_run_best.pt"
)

VISDRONE_VAL_DIR = PROJECT_ROOT / "datasets" / "VisDrone" / "images" / "val"
INFERENCE_DIR = PROJECT_ROOT / "outputs" / "inference"
RUN_NAME = "registered_visdrone_on_visdrone_val_samples"

device = 0 if torch.cuda.is_available() else "cpu"

print("Model:")
print(REGISTERED_VISDRONE_MODEL)

print("\nVisDrone validation directory:")
print(VISDRONE_VAL_DIR)

print("\nValidation directory exists:")
print(VISDRONE_VAL_DIR.exists())

if not VISDRONE_VAL_DIR.exists():
    raise FileNotFoundError(f"VisDrone validation directory not found: {VISDRONE_VAL_DIR}")

image_files = sorted(
    list(VISDRONE_VAL_DIR.glob("*.jpg")) +
    list(VISDRONE_VAL_DIR.glob("*.png")) +
    list(VISDRONE_VAL_DIR.glob("*.jpeg"))
)

print("\nTotal validation images found:")
print(len(image_files))

if len(image_files) == 0:
    raise FileNotFoundError("No validation images found.")

sample_images = image_files[:20]

print("\nTesting on first 20 validation images.")

model = YOLO(str(REGISTERED_VISDRONE_MODEL))

safe_label_map = {
    "pedestrian": "pedestrian",
    "people": "person",
    "bicycle": "vehicle",
    "car": "car",
    "van": "van",
    "truck": "truck",
    "tricycle": "vehicle",
    "awning-tricycle": "vehicle",
    "bus": "bus",
    "motor": "vehicle",
}

results = model.predict(
    source=[str(p) for p in sample_images],
    imgsz=640,
    conf=0.15,
    device=device,
    save=True,
    save_txt=True,
    save_conf=True,
    project=str(INFERENCE_DIR),
    name=RUN_NAME,
    exist_ok=True,
    verbose=True,
)

original_counts = Counter()
safe_counts = Counter()

for result in results:
    for box in result.boxes:
        class_id = int(box.cls[0].item())
        original_name = result.names[class_id]
        safe_name = safe_label_map.get(original_name, "object_for_review")

        original_counts[original_name] += 1
        safe_counts[safe_name] += 1

saved_dir = INFERENCE_DIR / RUN_NAME

print("\nInference complete.")
print("Saved directory:")
print(saved_dir)

print("\nOriginal VisDrone class counts:")
if original_counts:
    for name, count in original_counts.items():
        print(f"- {name}: {count}")
else:
    print("No detections")

print("\nSafe surveillance label counts:")
if safe_counts:
    for name, count in safe_counts.items():
        print(f"- {name}: {count}")
else:
    print("No detections")

print("\nOpen result folder:")
print(saved_dir)

Model:
/home/ronie/Programs/SWARM_DRONES/models/visdrone/yolov8n_visdrone_full_run_best.pt

VisDrone validation directory:
/home/ronie/Programs/SWARM_DRONES/datasets/VisDrone/images/val

Validation directory exists:
True

Total validation images found:
548

Testing on first 20 validation images.

0: 640x640 4 pedestrians, 10 peoples, 15 cars, 1 van, 1 truck, 3 tricycles, 1 awning-tricycle, 79 motors, 7.5ms
1: 640x640 6 pedestrians, 8 peoples, 18 cars, 2 vans, 13 motors, 7.5ms
2: 640x640 6 pedestrians, 12 peoples, 5 cars, 2 tricycles, 2 awning-tricycles, 29 motors, 7.5ms
3: 640x640 34 pedestrians, 5 peoples, 2 bicycles, 3 cars, 3 awning-tricycles, 31 motors, 7.5ms
4: 640x640 27 pedestrians, 3 peoples, 119 cars, 7 vans, 1 tricycle, 12 motors, 7.5ms
5: 640x640 3 pedestrians, 1 people, 82 cars, 6 vans, 3 trucks, 1 tricycle, 1 awning-tricycle, 13 motors, 7.5ms
6: 640x640 43 pedestrians, 2 peoples, 64 cars, 2 vans, 2 motors, 7.5ms
7: 640x640 34 pedestrians, 9 peoples, 1 bicycle, 14 cars, 1 v